# Model Experiments — MLflow Comparison

This notebook shows how to:
1. Load all experiments from MLflow
2. Compare models visually
3. Analyse Optuna hyperparameter importance
4. Inspect the best model's predictions



In [ ]:
import sys
sys.path.insert(0, '..')

import mlflow
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])
print(f'MLflow tracking URI: {config["mlflow"]["tracking_uri"]}')

## 1. Load All Experiment Runs

In [ ]:
experiment = mlflow.get_experiment_by_name(config['mlflow']['experiment_name'])

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.f1 DESC']
)

print(f'Total runs: {len(runs)}')

# Clean display
display_cols = [
    'tags.model_type', 'metrics.f1', 'metrics.accuracy',
    'metrics.roc_auc', 'metrics.precision', 'metrics.recall',
    'metrics.optuna_best_cv_f1', 'start_time'
]
display_cols = [c for c in display_cols if c in runs.columns]
runs[display_cols].rename(columns=lambda x: x.replace('metrics.','').replace('tags.','')).head(20)

## 2. Model Comparison — All Metrics

In [ ]:
metrics = ['f1', 'accuracy', 'roc_auc', 'precision', 'recall']

# Best run per model type
best_per_model = (
    runs.groupby('tags.model_type')
    .apply(lambda g: g.nlargest(1, 'metrics.f1'))
    .reset_index(drop=True)
)

# Radar / bar comparison
fig = go.Figure()
colors = {'logistic_regression': '#6366f1', 'xgboost': '#f59e0b', 'lightgbm': '#22c55e'}

for _, row in best_per_model.iterrows():
    model_name = row.get('tags.model_type', 'unknown')
    vals = [row.get(f'metrics.{m}', 0) for m in metrics]
    fig.add_trace(go.Bar(
        name=model_name,
        x=metrics,
        y=vals,
        marker_color=colors.get(model_name, '#94a3b8'),
        text=[f'{v:.3f}' for v in vals],
        textposition='outside'
    ))

fig.update_layout(
    barmode='group',
    title='📊 Best Model Comparison — All Metrics',
    height=450,
    plot_bgcolor='rgba(0,0,0,0)',
    yaxis=dict(range=[0, 1.05])
)
fig.show()

## 3. F1 Score Across All Runs (Learning Curve)

In [ ]:
if 'tags.model_type' in runs.columns and 'metrics.f1' in runs.columns:
    runs_sorted = runs.sort_values('start_time').reset_index(drop=True)
    runs_sorted['run_index'] = range(len(runs_sorted))

    fig = px.scatter(
        runs_sorted,
        x='run_index',
        y='metrics.f1',
        color='tags.model_type',
        size='metrics.roc_auc',
        hover_data=['run_id', 'metrics.accuracy', 'metrics.roc_auc'],
        title='F1 Score Across All Runs (size = ROC-AUC)',
        labels={'run_index': 'Run #', 'metrics.f1': 'F1 Score'}
    )
    fig.add_hline(
        y=config['monitoring']['f1_threshold'],
        line_dash='dash', line_color='red',
        annotation_text=f'Threshold ({config["monitoring"]["f1_threshold"]})'
    )
    fig.update_layout(height=400, plot_bgcolor='rgba(0,0,0,0)')
    fig.show()

## 4. Load Best Model & Evaluate on Test Set

In [ ]:
import mlflow.pyfunc
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Load production model
model_uri = f"models:/{config['mlflow']['model_name']}/Production"
try:
    model = mlflow.pyfunc.load_model(model_uri)
    print(f'✅ Loaded: {model_uri}')
except Exception as e:
    print(f'⚠️  Could not load Production model: {e}')
    print('Run scripts/run_pipeline.py first to train and register a model.')
    model = None

In [ ]:
if model is not None:
    from sklearn.model_selection import train_test_split

    df_features = pd.read_parquet('../data/features/churn_features.parquet')
    target = config['data']['target_column']
    X = df_features.drop(columns=[target])
    y = df_features[target].astype(int)

    _, X_test, _, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    y_prob = model.predict(X_test)
    if hasattr(y_prob, 'ndim') and y_prob.ndim == 2:
        y_prob = y_prob[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

In [ ]:
if model is not None:
    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'ROC (AUC = {roc_auc:.3f})',
        line=dict(color='#6366f1', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines',
        name='Random', line=dict(color='gray', dash='dash')
    ))
    fig.update_layout(
        title=f'ROC Curve — Production Model (AUC={roc_auc:.3f})',
        xaxis_title='False Positive Rate',
        yaxis_title='True Positive Rate',
        height=400, plot_bgcolor='rgba(0,0,0,0)'
    )
    fig.show()

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    fig2 = px.imshow(
        cm,
        labels=dict(x='Predicted', y='Actual', color='Count'),
        x=['Stayed', 'Churned'], y=['Stayed', 'Churned'],
        text_auto=True, color_continuous_scale='Blues',
        title='Confusion Matrix'
    )
    fig2.update_layout(height=350)
    fig2.show()